# API-232 Lab: Does Getting the Ball First Win Overtime?
### A randomized experiment run by a coin toss

When an NFL game is tied after four quarters, it goes to overtime. A **coin toss** decides which team chooses first, and the winner almost always chooses to **receive the kickoff** — to get the ball first.

That makes overtime a natural **randomized experiment**:

* **Treatment:** your team gets the ball first in overtime.
* **Control:** the other team gets the ball first.
* **Outcome:** your team wins the game.
* **Assignment mechanism:** a coin flip.

The NFL has changed its overtime rules twice because it worried that the coin toss was deciding too many games:

| Rules | Seasons | How overtime ends |
|---|---|---|
| **Sudden death** | Regular season through 2011; playoffs through the 2009 season | First team to score any points wins. |
| **Modified sudden death** | Regular season 2012–2024; playoffs 2010–2021 | A touchdown on the first possession wins. A first-possession field goal gives the other team one possession. (The regular-season period was cut from 15 to 10 minutes in 2017.) |
| **Both teams get the ball** | Playoffs from 2022; regular season from 2025 | Both teams are guaranteed a possession, whatever happens on the first one. |

In this lab you will use real play-by-play data from every NFL game from 1999 to 2025 to answer:

1. Is getting the ball first an advantage? How big, and how sure are we?
2. Did the 2010/2012 rule change reduce the advantage?
3. Could this data even *detect* the change the NFL was hoping for?

**Concepts from Quiz 4:** treatment effects, randomization and balance, standard errors, confidence intervals, statistical significance vs. practical importance, power, and non-interference.

**Time:** about 60 minutes. Answer the written questions in the markdown cells marked **✏️ Your answer**.


## 0. Setup (about 3 minutes)

Run the next two cells. The first installs `nflreadpy`, the Python package for the free [nflverse](https://github.com/nflverse) NFL data. The second downloads one season at a time and keeps only overtime plays, so it stays within Colab's memory. **Downloading 27 seasons takes 1–3 minutes.**

In [ ]:
!pip install -q nflreadpy

import math
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import nflreadpy as nfl

pd.set_option("display.precision", 3)
rng = np.random.default_rng(232)  # fixed seed so everyone gets the same simulation results

In [ ]:
COLS = ["game_id", "play_id", "season", "season_type", "qtr", "play_type", "posteam",
        "home_team", "away_team", "home_score", "away_score", "spread_line",
        "fixed_drive", "fixed_drive_result"]

frames = []
for season in range(1999, 2026):
    pbp = nfl.load_pbp(season)
    frames.append(pbp.filter(pl.col("qtr") >= 5).select(COLS).to_pandas())  # qtr 5 = overtime
    print(season, end=" ")

ot_plays = pd.concat(frames, ignore_index=True).sort_values(["game_id", "play_id"])
print(f"\nLoaded {len(ot_plays):,} overtime plays from {ot_plays.game_id.nunique()} games.")

### Building the game-level data (provided)

The cell below turns plays into **one row per overtime game**. The team that **received the opening overtime kickoff** is the team that got the ball first. Read the code and comments, but you don't need to change anything.

Variables you'll use:

* `received_first`: team that got the ball first (**treated** team)
* `first_wins`: 1 if the treated team won the game, 0 if it lost (ties are dropped)
* `era`: which overtime rules were in force
* `recv_home`: 1 if the treated team was the home team
* `recv_spread`: the treated team's expected margin of victory from the pregame betting line (positive = it was favoured)

In [ ]:
# The receiving team on the first overtime kickoff got the ball first.
kick = (ot_plays[ot_plays.play_type == "kickoff"]
        .groupby("game_id", as_index=False).first()[["game_id", "posteam"]]
        .rename(columns={"posteam": "received_first"}))

games = (ot_plays.groupby("game_id", as_index=False)
         .agg(season=("season", "first"), season_type=("season_type", "first"),
              home_team=("home_team", "first"), away_team=("away_team", "first"),
              home_score=("home_score", "max"), away_score=("away_score", "max"),
              spread_line=("spread_line", "first"))   # spread_line > 0 means the home team was favoured
         .merge(kick, on="game_id"))

games["margin"] = games.home_score - games.away_score
games["winner"] = np.select([games.margin > 0, games.margin < 0],
                            [games.home_team, games.away_team], default="TIE")

def rules(row):
    playoffs = row.season_type == "POST"
    if (not playoffs and row.season <= 2011) or (playoffs and row.season <= 2009):
        return "1. Sudden death"
    if (not playoffs and row.season <= 2024) or (playoffs and row.season <= 2021):
        return "2. Modified sudden death"
    return "3. Both teams get the ball"

games["era"] = games.apply(rules, axis=1)
games["recv_home"] = (games.received_first == games.home_team).astype(int)
games["recv_spread"] = np.where(games.recv_home == 1, games.spread_line, -games.spread_line)

print("Ties:", (games.winner == "TIE").sum())
ot = games[games.winner != "TIE"].copy()
ot["first_wins"] = (ot.winner == ot.received_first).astype(int)
print("Overtime games with a winner:", len(ot))
ot.head()

## Part 1. Was the coin toss really random? (about 10 minutes)

Before estimating any effect, we check **balance**: if the toss is random, the treated teams (got the ball first) should look like the control teams on things decided *before* the toss.

We'll check two pre-treatment characteristics:

1. **Home field.** In a random assignment, the home team should get the ball first about 50% of the time.
2. **Team strength.** The pregame point spread tells us which team the betting market expected to win. If the toss is random, the treated team's average expected margin should be about **0**.

**1a.** Compute the share of games in which the home team got the ball first, and the average of `recv_spread`. Report a standard error for each.

*Hint:* the standard error of a mean is the standard deviation divided by √n. For a share p, it is √(p(1 − p)/n).

In [ ]:
# TODO: share of games where the home team received first, and its standard error
n = len(ot)
home_share = ...
home_se = ...

# TODO: mean of recv_spread and its standard error (use .dropna(): a few early games have no betting line)
spread = ot.recv_spread.dropna()
spread_mean = ...
spread_se = ...

print(f"Home team got the ball first: {home_share:.3f} (SE {home_se:.3f})")
print(f"Receiving team's average expected margin: {spread_mean:+.2f} points (SE {spread_se:.2f})")

**1b.** Are both results consistent with a random toss? Explain using the standard errors.

✏️ **Your answer:**

&nbsp;

**1c.** The data doesn't record *who won the toss*, only *who got the ball first*. The toss winner chooses, and almost always picks the ball. In 2002, though, the Detroit Lions won an overtime toss and chose to kick off into the wind instead. They lost without touching the ball.

Our "treatment" is *getting the ball first*, not *winning the toss*. When could that difference bias our estimate? Connect your answer to the coaching example in Quiz 4, Question 2(d).

✏️ **Your answer:**


## Part 2. How big is the first-possession advantage? (about 15 minutes)

**2a.** Using all overtime games, compute:

* the share of games won by the team that got the ball first, `p_hat`
* its standard error
* a 95% confidence interval
* the z-score for the null hypothesis that getting the ball first doesn't matter (p = 0.5)
* the two-sided p-value

Use 1.96 for the confidence interval. The two-sided p-value for a z-score is `math.erfc(abs(z) / math.sqrt(2))`.

In [ ]:
# TODO
p_hat = ...
se = ...
ci_low, ci_high = ..., ...
z = ...
p_value = ...

print(f"First possession wins {p_hat:.3f} of games (n = {len(ot)})")
print(f"SE = {se:.3f}; 95% CI = [{ci_low:.3f}, {ci_high:.3f}]; z = {z:.2f}; p = {p_value:.3f}")

**2b. From a win rate to a treatment effect.** In every game, one team is treated and one is control. If treated teams win a share p of games, control teams win 1 − p.

(i) Write the average treatment effect of getting the ball first on the probability of winning, in terms of p. Compute it.

(ii) A classmate computes the standard error of this effect with the Quiz 4 formula SE = √(s²ₜ/nₜ + s²꜀/n꜀), treating the treated and control teams as two independent groups. Why is that wrong here? (*Hint:* if the treated team wins, what happens to the control team? Which Quiz 4 assumption does that relate to?)

✏️ **Your answer:**

&nbsp;

**2c.** Which statement correctly interprets your 95% confidence interval from 2a? Explain your choice in one sentence.

1. There is a 95% chance that the true first-possession win rate is in this interval.
2. If we could rerun overtime many times with new coin tosses, about 95% of intervals built this way would contain the true win rate.
3. About 95% of overtime games are won by the team that gets the ball first.
4. If we reran overtime many times, about 95% of estimated win rates would fall in this interval.

✏️ **Your answer:**


## Part 3. Did the rule change work? (about 15 minutes)

The NFL brought in modified sudden death to reduce the coin toss's influence. Under sudden death, a field goal on the first possession ended the game. Under the new rules it doesn't.

**3a.** Fill in the function below, then use it to build a table by `era` with n, the win rate, SE, 95% CI and z-score. The cell after it plots the results.

In [ ]:
def summarize(x):
    """x is a Series of 0/1 outcomes. Return n, p, SE, 95% CI and the z-score against 0.5."""
    n = len(x)
    p = ...       # TODO
    se = ...      # TODO
    return pd.Series({"n": n, "p": p, "se": se,
                      "ci_low": p - 1.96 * se, "ci_high": p + 1.96 * se,
                      "z": (p - 0.5) / se})

by_era = ot.groupby("era").first_wins.apply(summarize).unstack()
by_era

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
y = np.arange(len(by_era))
ax.errorbar(by_era.p, y, xerr=1.96 * by_era.se, fmt="o", capsize=4)
ax.axvline(0.5, color="grey", ls="--", lw=1)
ax.set_yticks(y, [f"{era}  (n={int(n)})" for era, n in zip(by_era.index, by_era.n)])
ax.invert_yaxis()
ax.set_xlabel("Share of overtime games won by the team that got the ball first")
ax.set_title("First-possession win rate with 95% confidence intervals")
plt.tight_layout(); plt.show()

**3b.** Estimate the change in the first-possession win rate from sudden death to modified sudden death, with a standard error, a 95% CI and a z-score.

*Hint:* the two eras are different games, so here the samples **are** independent: SE(difference) = √(SE₁² + SE₂²).

In [ ]:
# TODO
sd = by_era.loc["1. Sudden death"]
mod = by_era.loc["2. Modified sudden death"]

diff = ...
se_diff = ...
z_diff = ...

print(f"Change: {diff:+.3f} (SE {se_diff:.3f}); 95% CI = [{diff - 1.96*se_diff:+.3f}, {diff + 1.96*se_diff:+.3f}]; z = {z_diff:.2f}")

**3c.** A sports columnist writes: *"The rule change didn't work: the first-possession advantage is still statistically significant."* A league official replies: *"The change between eras isn't statistically significant, so we can't say it did anything."* Evaluate both claims using your results.

✏️ **Your answer:**

&nbsp;

**3d.** Look at the "Both teams get the ball" era. What can, and can't, we conclude from it?

✏️ **Your answer:**


## Part 4. Could we have detected a rule-change effect? (about 12 minutes)

Suppose the rule change *really* cut the first-possession win rate by d percentage points. How likely is a study with our sample sizes to find a statistically significant drop? That probability is the study's **power**.

**4a.** Before simulating anything, use the Quiz 4 rule of thumb. Significance requires an estimate more than about 2 × SE(difference) from zero. With SE(difference) ≈ your answer from 3b, how big would the true drop need to be for **50% power**? For **80% power** (about 2.8 × SE)?

✏️ **Your answer:**

&nbsp;

**4b.** The simulation below checks your answer. For each possible true drop d, it creates 2,000 fake versions of history. In each one, sudden-death games are won at the observed sudden-death rate, modified games at that rate minus d, using the real number of games in each era. It then counts the share of runs in which a two-sided test at the 5% level detects a significant drop. Fill in the two TODO lines.

In [ ]:
n_sd, n_mod = int(sd.n), int(mod.n)
p_sd = sd.p
drops = np.arange(0, 0.31, 0.02)
power = []

for d in drops:
    p_mod = p_sd - d
    hits = 0
    for _ in range(2000):
        x_sd = rng.binomial(n_sd, p_sd) / n_sd       # simulated sudden-death win rate
        x_mod = rng.binomial(n_mod, p_mod) / n_mod   # simulated modified win rate
        se_sim = ...   # TODO: SE of the difference, using x_sd and x_mod
        z_sim = ...    # TODO: z-score for the change, x_mod - x_sd
        hits += z_sim < -1.96   # a significant DROP
    power.append(hits / 2000)

plt.figure(figsize=(6, 3.5))
plt.plot(drops * 100, power, marker="o")
plt.axhline(0.8, color="grey", ls="--", lw=1); plt.axhline(0.5, color="grey", ls=":", lw=1)
plt.xlabel("True drop in first-possession win rate (percentage points)")
plt.ylabel("Power"); plt.title("Power to detect the rule change")
plt.tight_layout(); plt.show()
pd.DataFrame({"true drop (pp)": drops * 100, "power": power}).T.round(2)

**4c.** Suppose the NFL's realistic goal was to cut the first-possession win rate by about **5 percentage points**. What is the power of our comparison to detect that? About how many overtime games per era would the comparison need for 80% power? (*Hint:* the SE shrinks with √n.)

✏️ **Your answer:**

&nbsp;

**4d.** Go back to the official's claim in 3c. In one or two sentences, how does the power analysis change how you'd respond?

✏️ **Your answer:**


## Part 5. Significance vs. importance (about 5 minutes)

**5.** Your 2b estimate says getting the ball first raises a team's win probability by roughly 15 percentage points or more. A coach says: *"That's not even statistically significant in some eras, so it doesn't matter."* A league executive says: *"Even if it's real, it's small."* Respond to both. Is the advantage *practically* important for a rule meant to make overtime fair?

✏️ **Your answer:**


## Extension (optional): the mechanism

Under sudden death, a field goal on the first possession won the game. The modified rules took that away. If the rules worked, the **first drive's result** should matter less.

**E1.** Run the cell below. It reports how each era's first overtime drive ended and how often the team that got the ball first won after each result. Under modified sudden death, what happens to teams that kick a field goal on the first drive? Does that support the idea that the rule change shifted *how* teams win rather than *whether* the first team wins?

In [ ]:
first_drive = (ot_plays[ot_plays.fixed_drive.notna()]
               .groupby("game_id", as_index=False).first()[["game_id", "fixed_drive_result"]])
fd = ot.merge(first_drive, on="game_id")
fd["result"] = fd.fixed_drive_result.replace({"Turnover on downs": "Turnover", "Opp touchdown": "Turnover"})
summary = (fd.groupby(["era", "result"]).first_wins.agg(games="size", first_team_win_rate="mean")
           .reset_index())
summary["share_of_era"] = summary.games / summary.groupby("era").games.transform("sum")
summary.sort_values(["era", "games"], ascending=[True, False])

✏️ **Your answer:**